# Guía de Práctica N° 10: Regresión lineal múltiple y métricas de evaluación
**Curso:** Análisis de Datos (SIS323) · Semestre Académico 2026-II  
**Datos:** `alquileres_puno_juliaca.csv`

## Paso 1: Dividir en entrenamiento y prueba (80 % / 20 %, `random_state=42`)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

alq = pd.read_csv("alquileres_puno_juliaca.csv")
X = pd.get_dummies(alq.drop(columns="precio_soles"), drop_first=True, dtype=int)
y = alq["precio_soles"]

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=42)

print("Predictoras:")
print(list(X.columns[:5]))
print(list(X.columns[5:]))
print("Entrenamiento:", X_ent.shape, "| prueba:", X_pru.shape)

Predictoras:
['area_m2', 'habitaciones', 'banos', 'distancia_centro_km', 'antiguedad_anios']
['ciudad_Puno', 'cochera_Sí', 'internet_fibra_Sí']
Entrenamiento: (256, 8) | prueba: (64, 8)


## Paso 2: Ajustar el modelo de regresión lineal múltiple

In [2]:
from sklearn.linear_model import LinearRegression

multiple = LinearRegression().fit(X_ent, y_ent)
coef = pd.Series(multiple.coef_, index=X.columns).round(1)
print(f"Intersección: {multiple.intercept_:.1f}")
print(coef.to_string())

Intersección: 176.9
area_m2                 9.4
habitaciones            2.5
banos                  64.2
distancia_centro_km   -45.3
antiguedad_anios       -2.8
ciudad_Puno            74.6
cochera_Sí             76.3
internet_fibra_Sí      61.5


## Plantilla C · Paso 2: Interpretación de los coeficientes

| Predictora | Coeficiente | Interpretación (manteniendo constantes las demás) | ¿Signo esperado? |
| :--- | :---: | :--- | :---: |
| `area_m2` | +9.4 | Cada m² adicional suma unos S/ 9.4 al precio del alquiler, manteniendo constantes las demás variables. | Sí (+) |
| `habitaciones` | +2.5 | Cada habitación adicional suma unos S/ 2.5 al alquiler, manteniendo constantes las demás variables. | Sí (+) |
| `banos` | +64.2 | Cada baño adicional suma unos S/ 64.2 al alquiler, manteniendo constantes las demás variables. | Sí (+) |
| `distancia_centro_km` | -45.3 | Cada kilómetro más lejos del centro resta unos S/ 45.3 al alquiler, manteniendo constantes las demás variables. | Sí (−) |
| `antiguedad_anios` | -2.8 | Cada año adicional de antigüedad resta unos S/ 2.8 al alquiler, manteniendo constantes las demás variables. | Sí (−) |
| `ciudad_Puno` | +74.6 | Estar en Puno cuesta unos S/ 74.6 más que en Juliaca (categoría de referencia), manteniendo constantes las demás variables. | Sí (+) |
| `cochera_Sí` | +76.3 | Contar con cochera suma unos S/ 76.3 frente a no tenerla (referencia), manteniendo constantes las demás variables. | Sí (+) |
| `internet_fibra_Sí` | +61.5 | Contar con internet de fibra óptica suma unos S/ 61.5 frente a no tenerlo (referencia), manteniendo constantes las demás variables. | Sí (+) |

## Paso 3 y Paso 4: Calcular $R^2$ y RMSE en prueba y comparar con el modelo simple

In [3]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def evaluar(nombre, modelo, Xe, Xp):
    modelo.fit(Xe, y_ent)
    pred = modelo.predict(Xp)
    r2 = r2_score(y_pru, pred)
    rmse = np.sqrt(mean_squared_error(y_pru, pred))
    print(f"{nombre:28s} R² = {r2:.3f}  RMSE = S/ {rmse:.0f}")
    return pred

pred_simple = evaluar("Simple (solo el área)", LinearRegression(), X_ent[["area_m2"]], X_pru[["area_m2"]])
pred_mult = evaluar("Múltiple (todas)", LinearRegression(), X_ent, X_pru)
print(f"Precio medio en prueba: S/ {y_pru.mean():.0f}")

errores = y_pru - pred_mult
print(f"RMSE calculado a mano: {np.sqrt((errores ** 2).mean()):.1f}")
print(f"Error absoluto medio (MAE): {errores.abs().mean():.1f}")

Simple (solo el área)        R² = 0.707  RMSE = S/ 119
Múltiple (todas)             R² = 0.838  RMSE = S/ 88
Precio medio en prueba: S/ 843
RMSE calculado a mano: 88.2
Error absoluto medio (MAE): 72.0


## Paso 5: Validación cruzada de cinco particiones (`cv=5`)

In [4]:
from sklearn.model_selection import cross_val_score

r2_cv_mult = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")
print("Múltiple - R² en cada partición:", r2_cv_mult.round(3))
print(f"Múltiple - R² medio: {r2_cv_mult.mean():.3f} ± {r2_cv_mult.std():.3f}")

r2_cv_simple = cross_val_score(LinearRegression(), X[["area_m2"]], y, cv=5, scoring="r2")
print("Simple   - R² en cada partición:", r2_cv_simple.round(3))
print(f"Simple   - R² medio: {r2_cv_simple.mean():.3f} ± {r2_cv_simple.std():.3f}")

Múltiple - R² en cada partición: [0.76  0.846 0.881 0.867 0.81 ]
Múltiple - R² medio: 0.833 ± 0.044
Simple   - R² en cada partición: [0.583 0.758 0.625 0.662 0.644]
Simple   - R² medio: 0.655 ± 0.058


## Plantilla B · Pasos 3 a 5: Comparación de modelos

| Modelo | $R^2$ en prueba | RMSE en prueba (S/) | $R^2$ en validación cruzada (media ± desv.) | Conclusión |
| :--- | :---: | :---: | :---: | :--- |
| Simple (solo el área) | 0.707 | S/ 119 | 0.655 ± 0.058 | Explica ~70.7 % en prueba, pero comete un error típico mayor (S/ 119) y presenta mayor variabilidad entre particiones. |
| Múltiple (todas las predictoras) | 0.838 | S/ 88 | 0.833 ± 0.044 | Mejora sustancialmente: explica 83.8 % de la variabilidad en prueba, reduce el RMSE en ~S/ 31 y es más estable en validación cruzada. |

- **Precio medio en prueba y qué significa el RMSE frente a él:** El precio medio en prueba es **S/ 843**. Un RMSE de **S/ 88** representa aproximadamente el **10.5 %** del precio medio (error típico moderado), frente al **14.1 %** (S/ 119) del modelo simple.
- **¿Qué modelo recomendamos a la inmobiliaria y por qué?** Recomendamos el **modelo de regresión lineal múltiple**, porque generaliza mejor en datos no vistos ($R^2 = 0.838$ en prueba y $0.833 \pm 0.044$ en validación cruzada), reduce el error típico de predicción en ~S/ 31 y captura factores clave del mercado (ciudad, distancia al centro, baños, cochera, fibra óptica y antigüedad).

## Paso 6 y Plantilla A: Documentación de decisiones de modelado

| Decisión | Qué hicimos y por qué |
| :--- | :--- |
| Variable objetivo y predictoras | **Objetivo:** `precio_soles`. **Predictoras (8):** `area_m2`, `habitaciones`, `banos`, `distancia_centro_km`, `antiguedad_anios`, `ciudad_Puno`, `cochera_Sí` e `internet_fibra_Sí`, porque todas aportan información relevante del mercado inmobiliario. |
| Codificación de las variables categóricas (categorías de referencia) | Usamos `pd.get_dummies(..., drop_first=True, dtype=int)` antes de dividir para convertir texto a variables indicadoras binarias (0/1) evitando colinealidad. **Referencias:** `Juliaca` (en `ciudad`), `No` (en `cochera`) y `No` (en `internet_fibra`). |
| División entrenamiento/prueba y semilla | `train_test_split(..., test_size=0.2, random_state=42)` obteniendo **256** filas para entrenamiento (80 %) y **64** para prueba (20 %). La semilla `42` garantiza reproducibilidad exacta. |
| Métricas elegidas y por qué | **$R^2$** en prueba para medir la proporción de variabilidad explicada en datos no vistos; **RMSE** (en soles) para conocer el tamaño típico del error en las unidades del precio; y **MAE** (S/ 72.0) como referencia menos sensible a errores grandes. |
| Validación cruzada (particiones) | `cross_val_score(..., cv=5, scoring="r2")` con 5 particiones sobre todo el conjunto para obtener una estimación estable del rendimiento ($0.833 \pm 0.044$) y medir su variabilidad. |